<a href="https://colab.research.google.com/github/K-miy/420-a57-sf/blob/main/Fairness.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install required libraries
!pip install aif360
!pip install 'aif360[Reductions]'
!pip install 'aif360[inFairness]'
!pip install tensorflow

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from aif360.datasets import BinaryLabelDataset
from aif360.metrics import BinaryLabelDatasetMetric, ClassificationMetric
from aif360.algorithms.preprocessing import Reweighing
from aif360.algorithms.inprocessing import AdversarialDebiasing
from aif360.algorithms.postprocessing import CalibratedEqOddsPostprocessing
import tensorflow.compat.v1 as tf
tf.reset_default_graph()  # Reset graph for re-runs
tf.disable_eager_execution()

In [ ]:
# Simulate biased data
np.random.seed(42)
n_samples = 100000
data = pd.DataFrame({
    'good_credit': np.random.choice([0, 1], size=n_samples, p=[0.4, 0.6]),
    'past_convictions': np.random.choice([0, 1], size=n_samples, p=[0.7, 0.3]),
    'immigrant': np.random.choice([0, 1], size=n_samples, p=[0.5, 0.5])
})

# Introduce bias: immigrants have lower good credit and more past convictions
data.loc[data['immigrant'] == 1, 'good_credit'] = np.random.choice([0, 1], size=len(data[data['immigrant'] == 1]), p=[0.7, 0.3])
data.loc[data['immigrant'] == 1, 'past_convictions'] = np.random.choice([0, 1], size=len(data[data['immigrant'] == 1]), p=[0.6, 0.4])

# Define probability of fraud based on conditions
def fraud_risk(row):
  prob = 0.1  # Base risk of fraud
  if row['good_credit'] == 0:  # Higher risk with bad credit
    prob += 0.3
  if row['past_convictions'] == 1:  # Higher risk with past convictions
    prob += 0.4
  return np.random.choice([0, 1], p=[1 - prob, prob])

# Generate 'bank_fraud' based on probabilities
data['bank_fraud'] = data.apply(fraud_risk, axis=1)


In [ ]:
# Split data
X = data.drop(columns=['bank_fraud'])
y = data['bank_fraud']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)


In [ ]:
# Convert data to AIF360 format
train = pd.concat([X_train, y_train], axis=1)
test = pd.concat([X_test, y_test], axis=1)

privileged_groups = [{'immigrant': 0}]
unprivileged_groups = [{'immigrant': 1}]


In [ ]:
# Convert data to BinaryLabelDataset
train_aif = BinaryLabelDataset(favorable_label=0, unfavorable_label=1,
                               df=train, label_names=['bank_fraud'],
                               protected_attribute_names=['immigrant'])

test_aif = BinaryLabelDataset(favorable_label=0, unfavorable_label=1,
                              df=test, label_names=['bank_fraud'],
                              protected_attribute_names=['immigrant'])



In [ ]:
# Compute fairness metrics function
def compute_fairness_metrics(dataset, predictions):
    metric = ClassificationMetric(
        dataset, predictions,
        unprivileged_groups=unprivileged_groups,
        privileged_groups=privileged_groups
    )
    return {
        'Accuracy': accuracy_score(dataset.labels, predictions.labels),
        'Disparate Impact': metric.disparate_impact(),
        'Demographic Parity': abs(metric.statistical_parity_difference()),
        'Equal Opportunity Diff': metric.equal_opportunity_difference(),
        'Average Odds Diff': metric.average_odds_difference()
    }

In [ ]:
# 1. Baseline Model
lr = LogisticRegression(solver='liblinear')
lr.fit(X_train, y_train)

baseline_pred = test_aif.copy()
baseline_pred.labels = lr.predict(X_test)
baseline_metrics = compute_fairness_metrics(test_aif, baseline_pred)

In [ ]:
# 2. Pre-processing: Unaware Model (Remove protected attribute 'immigrant')
X_train_unaware = X_train.drop(columns=['immigrant'])
X_test_unaware = X_test.drop(columns=['immigrant'])

lr_unaware = LogisticRegression(solver='liblinear')
lr_unaware.fit(X_train_unaware, y_train)

unaware_pred = test_aif.copy()
unaware_pred.labels = lr_unaware.predict(X_test_unaware)
unaware_metrics = compute_fairness_metrics(test_aif, unaware_pred)


In [ ]:
# 3. Pre-processing: Reweighing
rw = Reweighing(unprivileged_groups, privileged_groups)
rw.fit(train_aif)
train_transformed = rw.transform(train_aif)

lr_rw = LogisticRegression(solver='liblinear')
lr_rw.fit(train_transformed.features, train_transformed.labels.ravel())

rw_pred = test_aif.copy()
rw_pred.labels = lr_rw.predict(X_test)
rw_metrics = compute_fairness_metrics(test_aif, rw_pred)

In [ ]:
# 4. In-processing: Adversarial Debiasing
tf.reset_default_graph()  # Reset TensorFlow graph for re-runs
sess = tf.Session()
debiased_model = AdversarialDebiasing(privileged_groups, unprivileged_groups, scope_name='debiased_classifier', sess=sess)

debiased_model.fit(train_aif)

debias_pred = debiased_model.predict(test_aif)
debias_metrics = compute_fairness_metrics(test_aif, debias_pred)

In [ ]:
# 5. Post-processing: Calibrated Equalized Odds
cpp = CalibratedEqOddsPostprocessing(privileged_groups, unprivileged_groups, seed=42)

cpp.fit(train_aif, train_aif)
cpp_pred = cpp.predict(debias_pred)
cpp_metrics = compute_fairness_metrics(test_aif, cpp_pred)

In [ ]:
# Display metrics
print("\nFairness Metrics:")
print("Baseline:", baseline_metrics)
print("Unaware:", unaware_metrics)
print("Reweighing:", rw_metrics)
print("Adversarial Debiasing:", debias_metrics)
print("Post-processing:", cpp_metrics)

